In [1]:
import cv2
import mediapipe as mp

In [2]:
# MediaPipe Pose modülü
mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

pose = mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5)

In [5]:
# Video yakalama
cap = cv2.VideoCapture(0)

In [6]:
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)  # Aynalama
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    # MediaPipe Pose ile işleme
    results = pose.process(rgb_frame)

    if results.pose_landmarks:
        # Landmark koordinatlarını al
        landmarks = results.pose_landmarks.landmark

        mp_drawing.draw_landmarks(
            frame,
            results.pose_landmarks,
            mp_pose.POSE_CONNECTIONS,
            landmark_drawing_spec=mp_drawing_styles.get_default_pose_landmarks_style()
        )
        
        # Burun (NOSE) ve sağ el (RIGHT_HAND_INDEX) koordinatları
        nose = landmarks[0]
        right_hand1 = landmarks[16]
        right_hand2 = landmarks[18]
        right_hand3 = landmarks[20]
        right_hand4 = landmarks[22]

        min_x=min(right_hand1.x,right_hand2.x,right_hand3.x,right_hand4.x)
        max_x=max(right_hand1.x,right_hand2.x,right_hand3.x,right_hand4.x)

        min_y=min(right_hand1.y,right_hand2.y,right_hand3.y,right_hand4.y)
        max_y=max(right_hand1.y,right_hand2.y,right_hand3.y,right_hand4.y)
        
        if nose.x>min_x and nose.x<max_x and nose.y>min_y and nose.y<max_y:
            cv2.putText(frame, "Buruna dokundu", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
                       

    # Görüntüyü göster
    cv2.imshow("MediaPipe Pose Algilama", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [7]:
# MediaPipe Holistic modülü
mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils

In [8]:
# Holistic modelini başlat
holistic = mp_holistic.Holistic(
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

In [10]:
# Video yakalama
cap = cv2.VideoCapture(0)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)  # Aynalama
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    # MediaPipe Holistic ile işleme
    results = holistic.process(rgb_frame)

    if results.pose_landmarks and results.right_hand_landmarks and results.face_landmarks:
        # Landmark koordinatlarını al
        nose = results.face_landmarks.landmark[1]  # NOSE_TIP: 1 numaralı landmark (burnun ucu)
        
        # Sağ elin işaret parmağı ucu
        right_index_finger_tip = results.right_hand_landmarks.landmark[8]  # Index Finger Tip

        # Koordinatları piksele çevir
        h, w, _ = frame.shape
        nose_x, nose_y = int(nose.x * w), int(nose.y * h)
        finger_x, finger_y = int(right_index_finger_tip.x * w), int(right_index_finger_tip.y * h)

        # Burnun ve parmağın merkezine daire çiz
        cv2.circle(frame, (nose_x, nose_y), 5, (0, 255, 0), -1)
        cv2.circle(frame, (finger_x, finger_y), 5, (255, 0, 0), -1)

        # Parmağın buruna yakınlığını ölç
        distance = ((nose_x - finger_x) ** 2 + (nose_y - finger_y) ** 2) ** 0.5

        if distance < 30:  # Mesafe eşiği (piksel cinsinden) - istersen değiştir
            cv2.putText(frame, "Buruna dokundu!", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
    
    # İskelet ve landmark çizimi
    #mp_drawing.draw_landmarks(frame, results.face_landmarks, mp_holistic.FACEMESH_TESSELATION)
    #mp_drawing.draw_landmarks(frame, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
    #mp_drawing.draw_landmarks(frame, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS)

    # Görüntüyü göster
    cv2.imshow("MediaPipe Holistic - Burun Dokunma Algilama", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()